# Particle-scaled flocking: 10,000 particles

This notebook adapts [the initial dynamics workflow](../dynamics/01_flocking_discovery.ipynb) with the original particle-dependent box calculation, $L=\mathrm{int}(2\sqrt{N}/\mathrm{density})$. At $N=10000$ and density parameter 0.5, the reflecting box is 400 × 400. The original random sequential `init_positions` function runs with the configured particle count and a freshly seeded generator; heading angles continue from that generator. The force and update routines are unchanged.

Continuous coarse-grained fields use uniform spatial grids and composite trapezoidal integration in x, y and saved time. The duration remains 12 time units. Box size and coarse-field spacing are separate settings: automatic box scaling changes L, while `cg_res=1` retains spacing 1. The former fixed-box hexagonal runs are superseded and preserved in `runs/` and [the historical notebook](history/fixed_box_hexagonal_01_updated_flocking.ipynb); their initialization failure at L=126 does not apply to the scaled L=400 case. See [README](README.md).

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import os, sys, json, hashlib, subprocess, uuid

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'flocking/particles.py').is_file())
HERE = REPO/'flocking/updatedparameters'
sys.path.insert(0,str(REPO/'src'))
sys.path.insert(0,str(HERE))
os.environ.setdefault('NUMBA_NUM_THREADS','2')
os.environ.setdefault('MPLBACKEND','Agg')
import numpy as np
import scipy
import numba
import PySPIDER
from workflow import default_config, resolve_config, initialize_simulator, analyze
from flocking_data import load_trajectories
assert Path(PySPIDER.__file__).resolve().is_relative_to(REPO/'src')
started = perf_counter()
print('Interpreter:',sys.executable)
print('Loaded PySPIDER:',PySPIDER.__file__)
print('NumPy:',np.__version__,'SciPy:',scipy.__version__,'Numba:',numba.__version__)

## Explicit settings and optional trajectory reuse

`config.json` is the complete editable configuration. A new run stores an exact resolved copy. With `TRAJECTORY_PATH = None`, generate fresh trajectories; otherwise provide a path relative to this directory (or set the `FLOCKING_TRAJECTORY` environment variable). Reuse requires `simulation.json` beside the trajectory, or an explicit `SIMULATION_METADATA_PATH`; it must identify the original timestep, box, particle count and boundary conditions. Reuse performs both analyses again without advancing the simulator.

The saved-frame interval is 0.2. Each of 12 domains uses 24 × 24 spatial nodes at spacing 1 and 25 original time frames, spanning 23 × 23 × 4.8 physical units. Spatial padding is 16, exceeding the quartic kernel support radius $4\sqrt{11}$. The kernel is $5(1-r^2/R^2)^4/(\pi R^2)$ for $r<R$, zero otherwise, with $R=4\sqrt{11}$. Density and velocity moments are divided by the mean number density, $10000/400^2 = 0.0625$. No temporal smoothing or trajectory interpolation is applied.

In [ ]:
CONFIG_PATH = HERE/'config.json'
config = json.loads(CONFIG_PATH.read_text())
assert set(config) == set(default_config()), 'Configuration keys differ from workflow defaults'
config = resolve_config(config)
TRAJECTORY_PATH = os.environ.get('FLOCKING_TRAJECTORY') or None
SIMULATION_METADATA_PATH = os.environ.get('FLOCKING_SIMULATION_METADATA') or None
RUN_DIR = HERE/'runs'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'_'+uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True,exist_ok=False)
(RUN_DIR/'config.json').write_text(json.dumps(config,indent=2,allow_nan=False)+'\n')
print(json.dumps(config,indent=2))
print('Run directory:',RUN_DIR)

In [ ]:
simulation_started = perf_counter()
if TRAJECTORY_PATH is None:
    simulator = initialize_simulator(config)
    trajectory_path = RUN_DIR/'trajectories.npy'
    simulator['run_to_T'](config['T'],config['save_every'],str(trajectory_path))
    simulation_metadata = {key:simulator[key] for key in ['N','L','density','a','v','m','beta','eps','dt','seed','PERIODIC_X','rc','r_min']}
    simulation_metadata.update(requested_T=config['T'],save_every=config['save_every'],
        initialization=config['initialization'],minimum_separation=config['minimum_separation'],
        source='Unchanged particles.py init_positions, forces, step and run_to_T; freshly seeded original random sequential initialization')
else:
    trajectory_path = Path(TRAJECTORY_PATH).expanduser()
    if not trajectory_path.is_absolute():
        trajectory_path = HERE/trajectory_path
    trajectory_path = trajectory_path.resolve(strict=True)
    metadata_path = Path(SIMULATION_METADATA_PATH).expanduser() if SIMULATION_METADATA_PATH else trajectory_path.parent/'simulation.json'
    if not metadata_path.is_absolute():
        metadata_path = HERE/metadata_path
    simulation_metadata = json.loads(metadata_path.read_text())
    assert simulation_metadata['N'] == config['N'] and simulation_metadata['L'] == config['L']
    assert simulation_metadata['dt'] == config['dt'] and simulation_metadata['PERIODIC_X'] == config['periodic_x']
    simulation_metadata = dict(simulation_metadata,reused_metadata_path=os.path.relpath(metadata_path,RUN_DIR))
simulation_seconds = perf_counter()-simulation_started
(RUN_DIR/'simulation.json').write_text(json.dumps(simulation_metadata,indent=2,allow_nan=False)+'\n')
trajectory_hash = hashlib.sha256(trajectory_path.read_bytes()).hexdigest()
trajectory = load_trajectories(trajectory_path,simulation_dt=simulation_metadata['dt'],box_side=simulation_metadata['L'])
assert trajectory['particle_count'] == config['N']
print('Trajectory:',trajectory_path)
print('Particle/component/time shape:',trajectory['positions'].shape)
print('Time range:',trajectory['times'][[0,-1]],'; saved interval:',trajectory['deltat'])
print('Simulation or selection seconds:',simulation_seconds)
speeds = np.linalg.norm(trajectory['velocities'],axis=1)
polar_order = np.linalg.norm(trajectory['velocities'].mean(axis=0),axis=0)/speeds.mean(axis=0)
print('Speed range:',speeds.min(),speeds.max())
print('Polar order first / last:',polar_order[0],polar_order[-1])

## Weak matrices and sparse regression

The scalar and vector libraries retain complexity 4, at most two coarse-grained factors per term, and per-factor derivative limits of one time derivative and two spatial derivatives. Terms are sorted by their printed names before matrix evaluation. The existing coordinate-aware dataset adapter is fixed to trapezoidal integration on every axis, including the physical spacing factors in $dx\,dy\,dt$. There is no quadrature comparison.

Regression uses the established truncated inverse iteration with backward/forward passes, physical column scales ($L=8$, $T=2$), unit row normalization and all rows for training. Its jump threshold is $(\gamma,\delta)=(1.5,10^{-8})$, with at most 10 active terms and 3 passes. The normalized training-residual cutoff is 0.002. A second independent dataset construction repeats domain sampling, field evaluation, matrix assembly and regression from the same trajectory and configuration.

In [ ]:
first = analyze(trajectory,config,RUN_DIR/'analysis')
second = analyze(trajectory,config,RUN_DIR/'repeat')
comparisons = {}
for key in first['arrays']:
    a,b = first['arrays'][key],second['arrays'][key]
    comparisons[key] = dict(equal=bool(np.array_equal(a,b)),shape=list(a.shape),
                            max_absolute_difference=float(np.max(np.abs(a-b))) if a.size else 0.0)
repeatability = dict(arrays=comparisons,
    libraries_equal=first['libraries']==second['libraries'],
    domains_and_nodes_equal=first['grids']==second['grids'],
    equations_equal=first['equations']==second['equations'])
repeatability['all_equal'] = all(item['equal'] for item in comparisons.values()) and all(
    repeatability[key] for key in ['libraries_equal','domains_and_nodes_equal','equations_equal'])
assert repeatability['all_equal'], 'Repeated analysis differs'
assert hashlib.sha256(trajectory_path.read_bytes()).hexdigest() == trajectory_hash
print('Matrix shapes:',[lib['shape'] for lib in first['libraries']])
print('Analysis seconds:',first['seconds'],'; repeat seconds:',second['seconds'])
print('Repeatability:',json.dumps(repeatability,indent=2))
for equation in first['equations']:
    print(equation['equation'],'; training residual =',equation['training_residual'])

## Reproducibility record and interpretation

The two analysis directories retain unscaled physical weak matrices, column scales, dense coefficients aligned with ordered terms, exact integration corners, nodes and relations. `results.json` records the trajectory checksum, simulation parameters, complete regression settings, source hashes, loaded module paths and versions, Git state and timings. Notebook source-cell hashes exclude changing execution outputs.

These are exploratory training relations. Continuity provides a kinematic check; it does not establish that the force-dependent flocking dynamics have been identified. Fixed particle speed imposes moment identities, while the random initial state and short 12-unit trajectory affect interpretation. Independent trajectories and parameter sensitivity are needed for a dynamical claim.

In [ ]:
import nbformat
from scipy.spatial import cKDTree
source_paths = sorted((REPO/'src/PySPIDER').rglob('*.py')) + [HERE/'workflow.py',HERE/'config.json',HERE/'test_workflow.py',HERE.parent/'particles.py',HERE.parent/'dynamics/flocking_data.py',HERE.parent/'dynamics/flocking_quadrature.py']
notebook = nbformat.read(HERE/'01_updated_flocking.ipynb',as_version=4)
notebook_source = [dict(cell_type=c.cell_type,source=c.source) for c in notebook.cells]
initial_positions = trajectory['positions'][:,:,0]
summary = dict(created_utc=datetime.now(timezone.utc).isoformat(),
    config=config,simulation=simulation_metadata,particle_count=trajectory['particle_count'],box_side=trajectory['box_side'],
    snapshots_used=len(trajectory['times']),saved_steps=trajectory['steps'].tolist(),times=trajectory['times'].tolist(),
    saved_frame_dt=trajectory['deltat'],dropped_final_step=trajectory['dropped_final_step'],
    trajectory_path=os.path.relpath(trajectory_path,RUN_DIR),trajectory_sha256=trajectory_hash,
    initial_minimum_distance=float(cKDTree(initial_positions).query(initial_positions,k=2)[0][:,1].min()),
    integration_measure='physical dx dy dt; uniform samples; trapezoidal on all axes',
    regression=dict(scaler='physical column scales',unit_rows=True,train_fraction=1,initializer='power',
                    backward_forward=True,residual_type='matrix_relative',threshold_type='jump',gamma=1.5,delta=1e-8),
    libraries=first['libraries'],domains=first['grids'],matrix_shapes=[lib['shape'] for lib in first['libraries']],
    equations=first['equations'],repeatability=repeatability,
    speed_range=[float(speeds.min()),float(speeds.max())],polar_order=[float(polar_order[0]),float(polar_order[-1])],
    seconds=dict(simulation_or_selection=simulation_seconds,analysis=first['seconds'],repeat=second['seconds'],total=perf_counter()-started),
    interpreter=sys.executable,numpy_version=np.__version__,scipy_version=scipy.__version__,numba_version=numba.__version__,
    numba_threads=numba.get_num_threads(),loaded_modules={module.__name__:str(Path(module.__file__).resolve()) for module in [np,scipy,numba,PySPIDER]},
    git_head=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip(),
    git_status=subprocess.run(['git','status','--short'],cwd=REPO,capture_output=True,text=True,check=True).stdout,
    source_sha256={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    notebook_source_sha256=hashlib.sha256(json.dumps(notebook_source,sort_keys=True).encode()).hexdigest(),
    validation='Training relations only; continuity is not validation of identified force-dependent dynamics.')
(RUN_DIR/'results.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,allow_nan=False)+'\n')
assert summary['particle_count']==10000 and summary['box_side']==400
assert summary['snapshots_used']==61 and summary['repeatability']['all_equal']
assert summary['equations']
print('FULL_10000_PARTICLE_ACCEPTANCE_PASSED',RUN_DIR)
print('Results:',RUN_DIR/'results.json')
print('Wall seconds:',summary['seconds'])